# Profiling ONNX Models
This notebook is a companion of chapter 10 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is about profiling and getting performance insights for a [GPT-2 small]((https://huggingface.co/openai-community/gpt2) model after conversion to the [ONNX](https://onnx.ai/) format and optimization. The same code applies to any other LLM and the insights building part is generic for any ML/DL ONNX model profiling analysis. No hardware acceleration needed.  
More details about the code can be found in the related book's chapter.

Install the missing dependencies in the Colab VM (only ONNX and the ONNX runtime, plus mlprodict (for profiling data aggregation and clean up only). Please see note later in this notebook about the mlprodict package installation in later versions of the Colab runtime.

In [1]:
!pip install onnx onnxruntime optimum[onnxruntime]

Import the required packages and classes.

In [2]:
import logging
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BatchEncoding, GPT2LMHeadModel

Download the GPT-2 small model and companion tokenizer from the HF's Hub.

In [3]:
model_name = "openai-community/gpt2"

model: GPT2LMHeadModel = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generate text to verify that the vanilla model has been downloaded and set up properly.

In [4]:
sample_prompt = 'Here is some text to encode Hello World'
inputs = tokenizer(sample_prompt, return_tensors="pt")
print("input tensors")
print(inputs)
print("input tensor shape")
print(inputs["input_ids"].size())

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print("output tensor")
print(logits)
print("output shape")
print(logits.shape)

input tensors
{'input_ids': tensor([[ 4342,   318,   617,  2420,   284, 37773, 18435,  2159]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
input tensor shape
torch.Size([1, 8])
output tensor
tensor([[[ -34.3027,  -33.9891,  -37.5683,  ...,  -42.6734,  -42.0399,
           -34.6136],
         [ -83.3065,  -82.9769,  -86.1204,  ...,  -89.8063,  -89.4546,
           -83.6084],
         [ -91.4901,  -92.5656,  -95.6423,  ...,  -96.6183,  -98.1545,
           -91.5266],
         ...,
         [ -92.8820,  -94.8433,  -98.9224,  ..., -101.4426, -103.2702,
           -95.7642],
         [ -72.6140,  -76.3407,  -79.7973,  ...,  -87.3300,  -85.7930,
           -77.7521],
         [-103.6147, -108.7898, -109.6276,  ..., -116.8557, -116.5565,
          -107.4467]]])
output shape
torch.Size([1, 8, 50257])


Export the model to ONNX.

In [5]:
input_ids: BatchEncoding = tokenizer(
    sample_prompt, add_special_tokens=True,
    return_attention_mask=False, return_tensors="pt"
)
for k, v in input_ids.items():
    input_ids[k] = v.type(dtype=torch.int32)
input_tensor = input_ids['input_ids']

onnx_model_path='gpt2onnx.onnx'
torch.onnx.export(
    model,
    args=(input_tensor,),
    f=onnx_model_path,
    export_params=True,        # Store trained weights inside the model file
    opset_version=18,          # Opset 18 is stable in 2026
    do_constant_folding=True,  # Keeps your optimization
    input_names=['input_ids'],
    output_names=['logits']
   # dynamo=True              # Uncomment if using PyTorch 2.4+ for better OPT support
)
_ = model.eval()

W0420 18:31:26.747000 4709 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0420 18:31:26.749000 4709 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0420 18:31:26.750000 4709 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0420 18:31:26.751000 4709 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `GPT2LMHeadModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GPT2LMHeadModel([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `GPT2LMHeadModel([...]` with `torch.export.export(..., strict=True)`...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/output_graph.py:1860: UserWarning: While compiling, we found certain side effects happened in the model.forward. Here are the list of potential sources you can double check: ['<unknown source>']
  warnings.warn(


[torch.onnx] Obtain model graph for `GPT2LMHeadModel([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'RuntimeError'>: Found <class 'transformers.cache_utils.DynamicCache'> in output, which is not a known type. If this type holds tensors, you need to register a pytree for it. See https://github.com/pytorch/functorch/issues/475 for a brief explanation why. If you don't need to register a pytree, please leave a comment explaining your use case and we'll make this more ergonomic to deal with

(Refer to the full stack trace above for more information.)

In [7]:
!optimum-cli export onnx --model openai-community/gpt2 --task text-generation-with-past opt_6.7b_onnx/

2026-04-20 18:39:05.957047: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776710345.979535    6982 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776710345.986941    6982 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776710346.005794    6982 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776710346.005833    6982 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776710346.005836    6982 computation_placer.cc:177] computation placer alr

In [23]:
# This ensures every weight chunk is downloaded
# onnx-community/Bonsai-1.7b-onnx
# onnx-community/tiny-gpt2-ONNX
!huggingface-cli download onnx-community/tiny-gpt2-ONNX --local-dir ./tiny-gpt-model

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 18 files:   0% 0/18 [00:00<?, ?it/s]Downloading 'onnx/model_fp16.onnx' to 'tiny-gpt-model/.cache/huggingface/download/onnx/lb-9-sF6iXlMiJFsswRTsP1WKb8=.6c35f6ae638bf656e3f16a76d98ffc9314da25e12593b7ddbc514a4863576a49.incomplete'

README.md: 1.95kB [00:00, 6.41MB/s]
Download complete. Moving file to tiny-gpt-model/README.md

onnx/model_fp16.onnx:   0% 0.00/3.47M [00:00<?, ?B/s]

onnx/model.onnx:   0% 0.00/6.82M [00:00<?, ?B/s]


onnx/model_bnb4.onnx:   0% 0.00/6.82M [00:00<?, ?B/s]Downloading 'generation_config.json' to 'tiny-gpt-model/.cache/huggingface/download/3EVKVggOldJcKSsGjSdoUCN1AyQ=.bd968f46ce39e754a4603c69f53a2a4a08a13dbb.incomplete'




.gitattributes: 1.52kB [00:00, 8.86MB/s]
Download complete. Moving file to tiny-gpt-model/.gitattributes
Fetching 18 files:   6% 1/18 [00:00<00:03,  5.15it/s]



merges.txt: 0.00B [00:00, ?B/s]Downloading 'onnx/model_int8.onnx' to 'tiny-gpt-model/.cache/

In [25]:
from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

model_path = "./tiny-gpt-model" # Use the local directory

# Load the 4-bit quantized version for the best CPU performance
model = ORTModelForCausalLM.from_pretrained(
    model_path,
    subfolder="onnx",           # Point to the folder containing the .onnx files
    file_name="model_q4.onnx",   # Point to the specific file inside that folder
    provider="CPUExecutionProvider"
)

# Remember to load the tokenizer from the same folder
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [31]:
from huggingface_hub import hf_hub_download

# This specifically fetches the path to the required file in your local cache
# Changed filename from 'decoder_model.onnx' to 'model.onnx' to fix the 404 error
onnx_model_path = hf_hub_download(
    repo_id="onnx-community/tiny-gpt2-ONNX",
    filename="model.onnx",
    subfolder="onnx"
)

print(f"Your ONNX model is located at: {onnx_model_path}")

onnx/model.onnx:   0%|          | 0.00/6.82M [00:00<?, ?B/s]

Your ONNX model is located at: /root/.cache/huggingface/hub/models--onnx-community--tiny-gpt2-ONNX/snapshots/a02528eb97fc2e4c9f95a2c8d322e63b90d2fd89/onnx/model.onnx


Define a custom function to prepare the input for the ONNX model to run text generation for profiling.

In [35]:
def get_example_inputs(prompt_text, tokenizer, num_layer, device='cpu'):
    encodings_dict = tokenizer.batch_encode_plus(prompt_text, padding=True)

    # Change dtype to int64 to match ONNX expected input types
    input_ids = torch.tensor(encodings_dict["input_ids"], dtype=torch.int64)
    attention_mask = torch.tensor(encodings_dict["attention_mask"], dtype=torch.int64)
    position_ids = attention_mask.cumsum(-1) - 1
    position_ids.masked_fill_(position_ids < 0, 0)
    position_ids = position_ids.to(torch.int64)

    empty_past = []
    batch_size = input_ids.size(0)
    past_shape = [2, batch_size, num_attention_heads, 0, hidden_size // num_attention_heads]
    for i in range(num_layer):
        empty_past.append(torch.empty(past_shape).type(torch.float32).to(device))

    return input_ids, attention_mask, position_ids, empty_past

Collect some vanilla model specs that are rquired for preparing the input for the ONNX version.

In [32]:
num_layer = model.config.n_layer
num_attention_heads = model.config.n_head
hidden_size = model.config.n_embd

Run text generation using the ONNX model with profiling enabled.



In [42]:
import onnxruntime
import numpy as np

tokenizer.pad_token = tokenizer.eos_token
input_ids, attention_mask, position_ids, empty_past = get_example_inputs(['Here is some text to encode Hello World'], tokenizer, num_layer)

so = onnxruntime.SessionOptions()
so.enable_profiling = True
session = onnxruntime.InferenceSession(onnx_model_path, so, providers=["CPUExecutionProvider"])

# Prepare all required inputs for the session
ort_inputs = {
    "input_ids": np.ascontiguousarray(input_ids.cpu().numpy()),
    "attention_mask": np.ascontiguousarray(attention_mask.cpu().numpy()),
    "position_ids": np.ascontiguousarray(position_ids.cpu().numpy()),
}

# Map the empty past_key_values (KV cache) to the input feed
for i, past in enumerate(empty_past):
    # The function get_example_inputs returns a list of tensors where each tensor
    # has shape [2, batch, heads, sequence, features]
    ort_inputs[f"past_key_values.{i}.key"] = np.ascontiguousarray(past[0].cpu().numpy())
    ort_inputs[f"past_key_values.{i}.value"] = np.ascontiguousarray(past[1].cpu().numpy())

ort_outputs = session.run(None, ort_inputs)

Close the inference session and collect the profiling data. The `prof` variable contains the name of the generated JSON file.

In [43]:
prof = session.end_profiling()

# Model Optimization

Set up the logging level to see in the output which kind of optimizations are automatically applied.

In [38]:
logging.basicConfig()
logging.getLogger().setLevel(logging.INFO)

Optimize the model using the ONNX's native optimizer.

In [41]:
from onnxruntime.transformers import optimizer

onnx_optim_model_path="onnx-community/tiny-gpt2-ONNX"
# Setting opt_level=0 or adjusting optimization_options can help resolve DAG errors
optimized_model = optimizer.optimize_model(onnx_model_path,
                                           model_type='gpt2',
                                           num_heads=num_attention_heads,
                                           hidden_size=hidden_size,
                                           use_gpu=False,
                                           opt_level=0,
                                           verbose=True)
optimized_model.convert_float_to_float16()
optimized_model.save_model_to_file(onnx_optim_model_path)

INFO:fusion_base:Fused LayerNormalization: 9
INFO:fusion_base:Fused FastGelu: 4
INFO:onnx_model:Removed 60 nodes
INFO:onnx_model:Removed 108 nodes
INFO:onnx_model_gpt2:postprocess: remove Reshape count: 24
INFO:fusion_base:Fused FastGelu(add bias): 4
INFO:onnx_model_bert:opset version: 14


failed in shape inference <class 'Exception'>


RuntimeError: Graph is not a DAG: len(sorted_node_set)=80, len(graph.node)=227, failed at node FastGelu_AddBias_3

Run text generation using the ONNX optimized model with profiling enabled.

In [ ]:
import onnx

optimized_onnx_model = onnx.load(onnx_optim_model_path)

tokenizer.pad_token = tokenizer.eos_token
input_ids, attention_mask, position_ids, empty_past = get_example_inputs(
    ['Here is some text to encode Hello World'], tokenizer, num_layer)

so = onnxruntime.SessionOptions()
so.enable_profiling = True
session = onnxruntime.InferenceSession(onnx_optim_model_path, so,
                                       providers=["CPUExecutionProvider"])
ort_inputs = {
    "input_ids": np.ascontiguousarray(input_ids.cpu().numpy()),
}
ort_outputs = session.run(None, ort_inputs)
prof_optimized = session.end_profiling()

# Profiling Data Clean Up and Visualization

Copying and pasting here the original *mlprodict*'s `OnnxWholeSession` class code as the installation of this package is failing on the latest version of the Colab runtime.

In [44]:
import json
import numpy

class OnnxWholeSession:
    """
    Runs the prediction for a single :epkg:`ONNX`,
    it lets the runtime handle the graph logic as well.

    :param onnx_data: :epkg:`ONNX` model or data
    :param runtime: runtime to be used, mostly :epkg:`onnxruntime`
    :param runtime_options: runtime options
    :param device: device, a string `cpu`, `cuda`, `cuda:0`...

    .. versionchanged:: 0.8
        Parameter *device* was added.
    """

    def __init__(self, onnx_data, runtime, runtime_options=None, device=None):
        if runtime not in ('onnxruntime1', 'onnxruntime1-cuda'):
            raise NotImplementedError(  # pragma: no cover
                f"runtime '{runtime}' is not implemented.")

        from onnxruntime import (  # delayed
            InferenceSession, SessionOptions, RunOptions,
            GraphOptimizationLevel)
        from onnxruntime.capi._pybind_state import (  # pylint: disable=E0611
            Fail as OrtFail, InvalidGraph as OrtInvalidGraph,
            InvalidArgument as OrtInvalidArgument,
            NotImplemented as OrtNotImplemented,
            RuntimeException as OrtRuntimeException)

        onnx_data0 = onnx_data
        if hasattr(onnx_data, 'SerializeToString'):
            onnx_data = onnx_data.SerializeToString()
        if isinstance(runtime_options, SessionOptions):
            sess_options = runtime_options
            session_options = None
            runtime_options = None
        else:
            session_options = (
                None if runtime_options is None
                else runtime_options.get('session_options', None))
            self.runtime = runtime
            sess_options = session_options or SessionOptions()
        self.run_options = RunOptions()
        self.run_options.log_severity_level = 3
        self.run_options.log_verbosity_level = 1

        if session_options is None:
            if runtime_options is not None:
                if runtime_options.get('disable_optimisation', False):
                    sess_options.graph_optimization_level = (  # pragma: no cover
                        GraphOptimizationLevel.ORT_ENABLE_ALL)
                if runtime_options.get('enable_profiling', True):
                    sess_options.enable_profiling = True
                if runtime_options.get('log_severity_level', 2) != 2:
                    v = runtime_options.get('log_severity_level', 2)
                    sess_options.log_severity_level = v
                    self.run_options.log_severity_level = v
        elif runtime_options is not None and 'enable_profiling' in runtime_options:
            raise RuntimeError(  # pragma: no cover
                "session_options and enable_profiling cannot be defined at the "
                "same time.")
        elif runtime_options is not None and 'disable_optimisation' in runtime_options:
            raise RuntimeError(  # pragma: no cover
                "session_options and disable_optimisation cannot be defined at the "
                "same time.")
        elif runtime_options is not None and 'log_severity_level' in runtime_options:
            raise RuntimeError(  # pragma: no cover
                "session_options and log_severity_level cannot be defined at the "
                "same time.")
        providers = ['CPUExecutionProvider']
        if runtime == 'onnxruntime1-cuda':
            providers = ['CUDAExecutionProvider'] + providers
        try:
            self.sess = InferenceSession(onnx_data, sess_options=sess_options,
                                         device=device, providers=providers)
        except (OrtFail, OrtNotImplemented, OrtInvalidGraph,
                OrtInvalidArgument, OrtRuntimeException, RuntimeError) as e:
            raise RuntimeError(
                "Unable to create InferenceSession due to '{}'\n{}.".format(e)) from e
        self.output_names = [_.name for _ in self.sess.get_outputs()]

    def run(self, inputs):
        """
        Computes the predictions.

        @param      inputs      dictionary *{variable, value}*
        @return                 list of outputs
        """
        v = next(iter(inputs.values()))
        if isinstance(v, (numpy.ndarray, dict)):
            try:
                return self.sess._sess.run(
                    self.output_names, inputs, self.run_options)
            except ValueError as e:
                raise ValueError(
                    "Issue running inference inputs=%r, expected inputs=%r."
                    "" % (
                        list(sorted(inputs)),
                        [i.name for i in self.sess.get_inputs()])) from e
        try:
            return self.sess._sess.run_with_ort_values(
                inputs, self.output_names, self.run_options)
        except RuntimeError:
            return self.sess._sess.run_with_ort_values(
                {k: v._get_c_value() for k, v in inputs.items()},
                self.output_names, self.run_options)

    @staticmethod
    def process_profiling(js):
        """
        Flattens json returned by onnxruntime profiling.

        :param js: json
        :return: list of dictionaries
        """
        rows = []
        for row in js:
            if 'args' in row and isinstance(row['args'], dict):
                for k, v in row['args'].items():
                    row[f'args_{k}'] = v
                del row['args']
            rows.append(row)
        return rows

    def get_profiling(self):
        """
        Returns the profiling informations.
        """
        prof = self.sess.end_profiling()
        with open(prof, 'r') as f:
            content = f.read()
        js = json.loads(content)
        return OnnxWholeSession.process_profiling(js)

Define a custom function to put the raw ONNX profiling data in a more friendly and useful format.

In [45]:
import json
import pandas as pd

def clean_up_profiling_data(prof):
  with open(prof, "r") as f:
      js = json.load(f)
  df = pd.DataFrame(OnnxWholeSession.process_profiling(js))

  return df

Define a custom function to do several profiling data aggregations (group by operator type and calculate the total duration for each one, count the number of occurrences for each one (and order them by duration), calculate the percentage of the total inference time for each one) that would be used to build some visualizations.

In [46]:
def transform_profiling_data_for_visualization(df):
  gr_dur = df[['dur', "args_op_name"]].groupby("args_op_name").sum().sort_values('dur')

  gr_n = df[['dur', "args_op_name"]].groupby("args_op_name").count().sort_values('dur')
  gr_n = gr_n.loc[gr_dur.index, :]

  gr_dur_perc = gr_dur / gr_dur['dur'].sum()

  return gr_dur, gr_n, gr_dur_perc

Transform the profiling data for the ONNX model.

In [47]:
gr_dur, gr_n, gr_dur_perc = transform_profiling_data_for_visualization(clean_up_profiling_data(prof))

Create visualizations for the ONNX model profiling data.

In [48]:
import plotly.express as px

fig = px.bar(gr_dur, x='dur',
             labels={
                     "dur": "Duration (ms)",
                     "args_op_name": "Operation type",
                 },
             title='Duration')
fig.show()

In [49]:
fig = px.bar(gr_n, x='dur',
             labels={
                     "dur": "Op count",
                     "args_op_name": "Operation type",
                 },
             title='Occurrences')
fig.show()

In [52]:
# Selected cell(s): a2LpQbd5Kinz
fig = px.bar(gr_dur_perc, x='dur',
             labels={
                     "dur": "Duration (%)",
                     "args_op_name": "Operation type",
                 },
             title='Proportion')
fig.show()

print("""
### Explanation of High-Latency Operations:

1. **Gather (Token Embedding & Indexing):**
   - This is primarily used for the **Embedding layer**. It 'gathers' the weight vector corresponding to a specific token ID from the vocabulary matrix.
   - In transformer blocks, it's also used for shuffling data or selecting specific indices during attention.

2. **Unsqueeze (Dimension Alignment):**
   - This adds a 'singleton' dimension (e.g., turning a [10] vector into a [1, 10] matrix).
   - While mathematically simple, GPT-2 uses many of these to align tensors for Broad-casting before Matrix Multiplications or to prepare inputs for the KV-cache.

3. **Concat (KV-Cache Management):**
   - This is often the most expensive 'utility' op. In every generation step, the model must **concatenate** the new Key/Value tensors with the 'Past Key/Values' from previous steps.
   - Since this requires allocating new memory and copying data, it becomes a bottleneck as the sequence length grows.
""")


### Explanation of High-Latency Operations:

1. **Gather (Token Embedding & Indexing):** 
   - This is primarily used for the **Embedding layer**. It 'gathers' the weight vector corresponding to a specific token ID from the vocabulary matrix. 
   - In transformer blocks, it's also used for shuffling data or selecting specific indices during attention.

2. **Unsqueeze (Dimension Alignment):**
   - This adds a 'singleton' dimension (e.g., turning a [10] vector into a [1, 10] matrix). 
   - While mathematically simple, GPT-2 uses many of these to align tensors for Broad-casting before Matrix Multiplications or to prepare inputs for the KV-cache.

3. **Concat (KV-Cache Management):**
   - This is often the most expensive 'utility' op. In every generation step, the model must **concatenate** the new Key/Value tensors with the 'Past Key/Values' from previous steps. 
   - Since this requires allocating new memory and copying data, it becomes a bottleneck as the sequence length grows.



Transform the profiling data for the optimized ONNX model.

In [ ]:
gr_dur, gr_n, gr_dur_perc = transform_profiling_data_for_visualization(clean_up_profiling_data(prof_optimized))

Create visualizations for the optimized ONNX model profiling data.

In [ ]:
fig = px.bar(gr_dur, x='dur',
             labels={
                     "dur": "Duration (ms)",
                     "args_op_name": "Operation type",
                 },
             title='Duration')
fig.show()

In [ ]:
fig = px.bar(gr_n, x='dur',
             labels={
                     "dur": "Op count",
                     "args_op_name": "Operation type",
                 },
             title='Occurrences')
fig.show()

In [ ]:
fig = px.bar(gr_dur_perc, x='dur',
             labels={
                     "dur": "Duration (%)",
                     "args_op_name": "Operation type",
                 },
             title='Proportion')
fig.show()